# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NAJAM2005/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip install -q pandas numpy
import os, subprocess
import numpy as np, pandas as pd
from IPython.display import display

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "https://github.com/NAJAM2005/flyrank-ml-internship.git"], check=True)
    os.chdir("flyrank-ml-internship")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)   # used for checks only, never as a score input
print(df.shape, "| declining share:", round(df["is_declining"].mean(), 3))

(30000, 45) | declining share: 0.542


## 1. My rule and its reason codes

**My rule in plain words:** For the `decline_recovery` lane, a page scores higher when it was not updated in more than 22 days and has many impressions in the last 90 days. CTR gap is left out of the score (weight 0) because its signal check was MIXED.

**Reason codes (one per page, the factor adding the most to its score):**
- `STALE_PAGE` -> action `REFRESH_CONTENT`
- `HIGH_DEMAND_PAGE` -> action `EXPAND_AND_REFRESH`
- `CTR_BELOW_PEERS` -> action `REWRITE_TITLE_META` (unused, weight 0)

**Signal check 1: staleness** (`days_since_last_update`, behind FlyRank's refresh flags)
Verdict: MIXED (n = 30,000). Observed: the decline rate is not steady across age buckets (53.9%, 39.3%, 59.9%, 54.7%), but pages not updated in more than 22 days decline about 60% of the time versus about 51% for fresher pages. Only 4 buckets formed because many pages share 20-22 days. I use staleness as a simple flag (> 22 days).

**Signal check 2: CTR vs position** (`ctr` vs the median CTR at the same position, behind the CTR-fix logic)
Verdict: MIXED (n = 25,932; 4,068 pages had no position or impressions). Observed: the decline rate falls as CTR rises relative to peers (64.5%, 58.3%, 50.9%), but the lowest-CTR bucket is lower (58.6%), possibly because near-zero-click pages have little traffic left to lose (a hypothesis, not tested here). Because the pattern is not clean, CTR gap gets weight 0 in this baseline.

**Why demand is in the rule:** impressions show how much a refresh is worth to an editor, not whether a page will decline.

All numbers are observed and directional, for decision-support.

In [9]:
def bucket_check(d, col, expect, q=5):
    x = d[[col, "is_declining"]].dropna()
    x = x.assign(bucket=pd.qcut(x[col], q, duplicates="drop"))
    t = (x.groupby("bucket", observed=True)
           .agg(n=("is_declining", "size"), decline_rate=("is_declining", "mean")).round(3))
    print(f"{col}  (total n = {len(x)})")
    display(t)
    r, diffs = t["decline_rate"].values, np.diff(t["decline_rate"].values)
    if r.max() - r.min() < 0.03:
        v = "FALSE"
    elif (diffs >= 0).all() or (diffs <= 0).all():
        v = "CONFIRMED" if ((diffs >= 0).all() == (expect == "up")) else "OPPOSITE"
    else:
        v = "MIXED"
    print("Suggested verdict:", v, "(read the table before you accept it)\n")
    return v

# Signal 1: staleness (expect more stale -> more decline)
v1 = bucket_check(df, "days_since_last_update", expect="up")

# Signal 2: CTR vs peers at the same position (expect lower ratio -> more decline)
d = df[(df["avg_position"] > 0) & (df["impressions_90d"] > 0)].copy()
d["pos_bucket"] = pd.qcut(d["avg_position"], 10, duplicates="drop")
d["expected_ctr"] = d.groupby("pos_bucket", observed=True)["ctr"].transform("median")
d["ctr_ratio"] = d["ctr"] / d["expected_ctr"].replace(0, np.nan)
df["ctr_ratio"] = d["ctr_ratio"]
v2 = bucket_check(df, "ctr_ratio", expect="down")

days_since_last_update  (total n = 30000)


,n,decline_rate
bucket,,
"(0.999, 20.0]",15866,0.539
"(20.0, 22.0]",3564,0.393
"(22.0, 104.0]",10252,0.599
"(104.0, 373.0]",318,0.547


Suggested verdict: MIXED (read the table before you accept it)

ctr_ratio  (total n = 25932)


,n,decline_rate
bucket,,
"(-0.001, 0.364]",10447,0.586
"(0.364, 1.625]",5190,0.645
"(1.625, 3.684]",5122,0.583
"(3.684, 2500.0]",5173,0.509


Suggested verdict: MIXED (read the table before you accept it)



## 2. Build the ranked queue (writes the CSV)

Score = 0.60 x (not updated in more than 22 days) + 0.40 x demand percentile. Ranked highest first and written to `work/outputs/baseline_action_score.csv`.


In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
W = {"stale": 0.60, "demand": 0.40, "ctrgap": 0.0}

p_stale  = (df["days_since_last_update"] > 22).astype(float)
p_demand = np.log1p(df["impressions_90d"]).rank(pct=True)
p_ctrgap = (1 - df["ctr_ratio"].rank(pct=True)).fillna(0)

contrib = pd.DataFrame({"stale": W["stale"] * p_stale,
                        "demand": W["demand"] * p_demand,
                        "ctrgap": W["ctrgap"] * p_ctrgap})
df["score"] = contrib.sum(axis=1).round(4)
df["reason_code"] = contrib.idxmax(axis=1).map({"stale": "STALE_PAGE",
                                                "demand": "HIGH_DEMAND_PAGE",
                                                "ctrgap": "CTR_BELOW_PEERS"})
df["action"] = df["reason_code"].map({"STALE_PAGE": "REFRESH_CONTENT",
                                      "HIGH_DEMAND_PAGE": "EXPAND_AND_REFRESH",
                                      "CTR_BELOW_PEERS": "REWRITE_TITLE_META"})

SCORE_INPUTS = ["days_since_last_update", "impressions_90d"]

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
os.makedirs("work/outputs", exist_ok=True)
cols = ["rank", "content_id", "score", "reason_code", "action",
        "days_since_last_update", "impressions_90d"]
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote", len(queue), "rows")

base = df["is_declining"].mean()
for k in (10, 20, 50):
    print(f"Precision@{k}: {queue.head(k)['is_declining'].mean():.2f}  (base rate {base:.2f})")


wrote 30000 rows
Precision@10: 0.50  (base rate 0.54)
Precision@20: 0.40  (base rate 0.54)
Precision@50: 0.42  (base rate 0.54)


## 3. Top-10 review

(Top 10 chosen, as the card asks.) Format: action | why it is ranked here | what would make it wrong.

1. REFRESH_CONTENT | STALE_PAGE: updated 48 days ago, 497,727 impressions in 90d | wrong if the topic is evergreen, or it was edited without the date changing
2. REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 517,715 impressions in 90d | wrong if 104 days is a placeholder value and not a real age
3. REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 347,399 impressions in 90d | wrong if 104 days is a placeholder value and not a real age
4. REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 309,192 impressions in 90d | wrong if 104 days is a placeholder value and not a real age
5. REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 309,910 impressions in 90d | wrong if the page is already recovering, since the rule cannot see recent trend
6. REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 443,434 impressions in 90d | wrong if the page is already recovering, since the rule cannot see recent trend
7. REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 286,608 impressions in 90d | wrong if the impressions come from a few irrelevant queries
8. REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 295,097 impressions in 90d | wrong if the impressions come from a few irrelevant queries
9. REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 233,561 impressions in 90d | wrong if the topic is evergreen and age does not hurt it
10. REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 211,366 impressions in 90d | wrong if the topic is evergreen and age does not hurt it

Confidence note: low to medium, directional only. The rule uses two inputs (update age and impressions), so it cannot tell whether a page is already recovering. 5 of these 10 are labeled declining.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
WRONG_IF = {
    "STALE_PAGE": "the topic is evergreen and age doesn't hurt it, or it was edited without the date changing",
    "HIGH_DEMAND_PAGE": "the impressions come from a few irrelevant queries, so a refresh would not lift clicks",
    "CTR_BELOW_PEERS": "the page shows a SERP feature or branded query that lowers CTR whatever the title says",
}
top10 = queue.head(10)
display(top10[["rank", "content_id", "action", "reason_code", "score"]])
for _, r in top10.iterrows():
    print(f"#{int(r['rank'])} {r['action']} | {r['reason_code']}: updated {int(r['days_since_last_update'])} days ago, "
          f"{int(r['impressions_90d'])} impressions in 90d | wrong if {WRONG_IF[r['reason_code']]}")


,rank,content_id,action,reason_code,score
0,1,content_2cb567c3c89b,REFRESH_CONTENT,STALE_PAGE,1.0000
1,2,content_5fe46e04994d,REFRESH_CONTENT,STALE_PAGE,1.0000
2,3,content_2c2606c5d176,REFRESH_CONTENT,STALE_PAGE,0.9999
3,4,content_9532f197bbc8,REFRESH_CONTENT,STALE_PAGE,0.9999
4,5,content_cb112fce36be,REFRESH_CONTENT,STALE_PAGE,0.9999
5,6,content_2dba2b1f9536,REFRESH_CONTENT,STALE_PAGE,0.9999
6,7,content_b28d1efd668f,REFRESH_CONTENT,STALE_PAGE,0.9998
7,8,content_36ff89c8214e,REFRESH_CONTENT,STALE_PAGE,0.9998
8,9,content_813e88069237,REFRESH_CONTENT,STALE_PAGE,0.9997
9,10,content_c21024970297,REFRESH_CONTENT,STALE_PAGE,0.9997


#1 REFRESH_CONTENT | STALE_PAGE: updated 48 days ago, 497727 impressions in 90d | wrong if the topic is evergreen and age doesn't hurt it, or it was edited without the date changing
#2 REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 517715 impressions in 90d | wrong if the topic is evergreen and age doesn't hurt it, or it was edited without the date changing
#3 REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 347399 impressions in 90d | wrong if the topic is evergreen and age doesn't hurt it, or it was edited without the date changing
#4 REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 309192 impressions in 90d | wrong if the topic is evergreen and age doesn't hurt it, or it was edited without the date changing
#5 REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 309910 impressions in 90d | wrong if the topic is evergreen and age doesn't hurt it, or it was edited without the date changing
#6 REFRESH_CONTENT | STALE_PAGE: updated 104 days ago, 443434 impressions in 90d | wro

## 4. Weak picks + leakage check

**Weak picks:** 29 of the top 50 are not labeled declining (Precision@50 = 0.42), and the table below also flags low-demand picks (35 flagged in total). The rule only looks at update age and impressions, so it can rank a page high even if it is already recovering. In this sample, the highest-impression fifth of pages declines less often (about 55%) than the middle fifths (about 60-63%), so putting demand in the score pulls the top ranks toward pages that are less likely to be declining. I did not test demand as a signal before using it, which is a limit of this baseline. Also, 8,773 pages (29%) share exactly 104 days since last update, which may be a placeholder value (a hypothesis, not confirmed).

**Leakage check:** the score uses only `days_since_last_update` and `impressions_90d`. It does not use `trend_direction`, `trend_pct`, or any last-30d / prev-30d column. The declining label was used only to check signals and measure precision, never to build the score.

**Baseline for Week 5:** Precision@10 / @20 / @50 = 0.50 / 0.40 / 0.42 against a base rate of 0.54. This baseline does not beat the base rate, so a Week-5 model has an easy bar to clear. All numbers are observed and directional, for decision-support.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
FORBIDDEN = ["trend_direction", "trend_pct", "is_declining",
             "impressions_last_30d", "impressions_prev_30d", "clicks_last_30d",
             "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]
leaked = [c for c in SCORE_INPUTS if c in FORBIDDEN]
assert not leaked, f"Leak: {leaked}"
print("Score inputs:", SCORE_INPUTS, "-> no label-derived or future-window columns")

weak = queue.head(50)
weak = weak[(weak["is_declining"] == 0) | (weak["impressions_90d"] < weak["impressions_90d"].quantile(0.25))]
print(len(weak), "of the top 50 look weak (not declining, or low demand):")
display(weak[["rank", "content_id", "reason_code", "score", "impressions_90d", "is_declining"]].head(10))


Score inputs: ['days_since_last_update', 'impressions_90d'] -> no label-derived or future-window columns
35 of the top 50 look weak (not declining, or low demand):


,rank,content_id,reason_code,score,impressions_90d,is_declining
0,1,content_2cb567c3c89b,STALE_PAGE,1.0000,497727,0
5,6,content_2dba2b1f9536,STALE_PAGE,0.9999,443434,0
6,7,content_b28d1efd668f,STALE_PAGE,0.9998,286608,0
7,8,content_36ff89c8214e,STALE_PAGE,0.9998,295097,0
9,10,content_c21024970297,STALE_PAGE,0.9997,211366,0
10,11,content_b511d4bc4ad2,STALE_PAGE,0.9996,205915,0
11,12,content_d17681677e69,STALE_PAGE,0.9996,201584,0
12,13,content_a7427266c305,STALE_PAGE,0.9996,201111,0
15,16,content_bb5bd5f771dc,STALE_PAGE,0.9995,176296,0
16,17,content_c5063073d048,STALE_PAGE,0.9995,192205,0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.